# Kelly Criterion Simulator

This notebook provides an interactive simulation of bankroll growth using Kelly Criterion position sizing. Adjust the controls below to describe a betting or trading strategy, then select **Run Simulation** to generate and plot one possible sequence of outcomes.

## How to use the simulator

1. Run the code cell below to display the interactive controls.
2. Set the assumptions for the simulation:
   - **Win Rate:** probability that each bet wins.
   - **Win Payoff:** profit earned per unit allocated when a bet wins.
   - **Lose Payoff:** amount lost per unit allocated when a bet loses.
   - **Initial Capital:** bankroll available before the first bet.
   - **Number of Bets:** number of consecutive outcomes to simulate.
   - **Kelly Fraction:** proportion of the full-Kelly allocation to use; `1.0` is full Kelly, `0.5` is half Kelly, and `0.25` is quarter Kelly.
3. Optionally enter an integer in **Seed** to make the outcome sequence reproducible. Leave it blank to generate a new random seed.
4. Select **Run Simulation**.

The chart shows how capital changes after each bet. Below the chart, the notebook reports the seed used, the actual fraction allocated per bet, and the longest winning and losing streaks. To reproduce a randomly generated run, copy its reported seed into the **Seed** field and run the simulation again.

> This simulator is intended for education and experimentation. It assumes independent outcomes with constant probabilities and payoffs; real trading and betting conditions may differ substantially.

## Kelly Criterion Formula

For a bet with:

- $p$ = probability of winning
- $q = 1-p$ = probability of losing
- $b$ = profit earned per unit wagered on a win
- $a$ = amount lost per unit wagered on a loss

the generalized Kelly fraction is:

$$
f^* = \max\left(0,\frac{pb-qa}{ab}\right)
$$

Here, $f^*$ is the optimal fraction of current capital allocated to each bet.
The `max(0, ...)` term prevents betting when the expected edge is not positive.

When the entire wager is lost on an unsuccessful bet, $a=1$, and the formula simplifies to:

$$
f^* = \max\left(0,\frac{bp-q}{b}\right)
$$

This simulator also supports fractional Kelly sizing:

$$
f_{\text{used}} = c f^*
$$

where $c$ is the selected Kelly fraction. For example:

- $c=1$ uses full Kelly
- $c=0.5$ uses half Kelly
- $c=0.25$ uses quarter Kelly

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact_manual, FloatSlider, IntSlider, Text
from kelly_criterion_simulator import simulate_kelly

# Main simulation function with dynamic RNG
def kelly_simulation_ui_with_seed(win_rate, win_payoff, lose_payoff, initial_capital, num_bets, kelly_fraction, seed):

    seed_value = None if not seed.strip() else int(seed)

    (
        capital_history,
        max_win_streak,
        max_lose_streak,
        bet_fraction,
        used_seed
    ) = simulate_kelly(
        win_rate=win_rate,
        win_payoff=win_payoff,
        lose_payoff=lose_payoff,
        initial_capital=initial_capital,
        num_bets=num_bets,
        kelly_fraction=kelly_fraction,
        seed=seed_value
    )

    plt.figure(figsize=(12, 6))
    plt.plot(capital_history, label="Capital Over Time")
    plt.title(f"Kelly Criterion Simulation (Bet Fraction: {bet_fraction:.3f})")
    plt.xlabel("Number of Rounds")
    plt.ylabel("Capital")
    plt.ticklabel_format(style='plain')
    plt.axhline(y=initial_capital, color='orange', linestyle='--', label="Initial Capital")
    plt.legend()
    plt.grid()
    plt.show()

    print(f"Random Seed: {used_seed}")
    print(f"Kelly Bet Fraction: {bet_fraction:.3f}")
    print(f"Longest Winning Streak: {max_win_streak}")
    print(f"Longest Losing Streak: {max_lose_streak}")


# Set up interactive sliders with a manual “Run Simulation” button
# Interactive controls
interactive_controls = interact_manual(
    kelly_simulation_ui_with_seed,
    win_rate=FloatSlider(min=0.1, max=0.9, step=0.01, value=0.55, description='Win Rate'),
    win_payoff=FloatSlider(min=0.1, max=10, step=0.1, value=1, description='Win Payoff'),
    lose_payoff=FloatSlider(min=0.1, max=10, step=0.1, value=1, description='Lose Payoff'),
    initial_capital=IntSlider(min=100, max=100000, step=100, value=30000, description='Initial Capital'),
    num_bets=IntSlider(min=10, max=1000, step=10, value=100, description='Number of Bets'),
    kelly_fraction=FloatSlider(min=0.01, max=1, step=0.01, value=0.20, description='Kelly Fraction'),
    seed=Text(value='', placeholder='Blank for random', description='Seed'),
    manual_name='Run Simulation'
);

